# B8. 电商 Dashboard 数据处理 Notebook

> **配套模块**: [B8 电商 Dashboard](../paths/b-developers/b8-ecommerce-dashboard.md)
>
> **功能**: 多平台数据整合 + KPI 计算 + 异常检测 + 可视化
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-skills/blob/main/notebooks/b8-dashboard-demo.ipynb)

---

## 1. 安装依赖

In [ ]:
!pip install -q pandas numpy plotly

## 2. 加载多平台真实数据

加载 Amazon + Shopify 的真实日粒度数据；缺少文件或字段时明确停止。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DASHBOARD_CSV = Path('commerce-daily.csv')
if not DASHBOARD_CSV.is_file():
    raise FileNotFoundError('请上传真实多平台数据并命名为 commerce-daily.csv')
df = pd.read_csv(DASHBOARD_CSV)
required = {'date', 'platform', 'revenue', 'orders', 'units', 'ad_spend', 'ad_revenue', 'refunds', 'cogs', 'fba_fees'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Dashboard CSV 缺少必需列: {sorted(missing)}')
if df.empty:
    raise ValueError('Dashboard CSV 不能为空')
df['date'] = pd.to_datetime(df['date'], errors='raise')
numeric = sorted(required - {'date', 'platform'})
for column in numeric:
    df[column] = pd.to_numeric(df[column], errors='raise')
if (df[numeric] < 0).any().any():
    raise ValueError('金额、订单和件数不能为负数')
if df.duplicated(['date', 'platform']).any():
    raise ValueError('同一 date + platform 只能有一行')
if not {'amazon', 'shopify'}.issubset(set(df['platform'].str.lower())):
    raise ValueError('数据必须同时包含 amazon 和 shopify')
df['net_profit'] = df['revenue'] - df['cogs'] - df['fba_fees'] - df['ad_spend'] - df['refunds']
df['acos'] = (df['ad_spend'] / df['ad_revenue'].replace(0, 1)) * 100
df['roas'] = df['ad_revenue'] / df['ad_spend'].replace(0, 1)

print(f'Total records: {len(df)}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Platforms: {df["platform"].unique()}')
df.head()

## 3. KPI 计算

In [ ]:
# Overall KPIs
total = df.groupby('platform').agg({
    'revenue': 'sum',
    'orders': 'sum',
    'ad_spend': 'sum',
    'ad_revenue': 'sum',
    'net_profit': 'sum',
    'refunds': 'sum'
}).round(0)

total['aov'] = (total['revenue'] / total['orders']).round(2)
total['roas'] = (total['ad_revenue'] / total['ad_spend']).round(1)
total['acos'] = ((total['ad_spend'] / total['ad_revenue']) * 100).round(1)
total['margin'] = ((total['net_profit'] / total['revenue']) * 100).round(1)
total['refund_rate'] = ((total['refunds'] / total['revenue']) * 100).round(1)

print('=== Platform KPI Summary (90 days) ===')
display(total[['revenue', 'orders', 'aov', 'roas', 'acos', 'margin', 'refund_rate']])

print(f'\nTotal Revenue: ${total["revenue"].sum():,.0f}')
print(f'Total Profit: ${total["net_profit"].sum():,.0f}')
print(f'Overall Margin: {total["net_profit"].sum() / total["revenue"].sum() * 100:.1f}%')

## 4. 异常检测

In [ ]:
def detect_anomalies(series, window=7, threshold=2.0):
    """Z-Score based anomaly detection"""
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()
    z_scores = (series - rolling_mean) / rolling_std
    return z_scores, abs(z_scores) > threshold

# Check Amazon revenue anomalies
amz = df[df['platform'] == 'amazon'].copy().reset_index(drop=True)
z_scores, is_anomaly = detect_anomalies(amz['revenue'])

anomalies = amz[is_anomaly]
print(f'=== Anomalies Detected: {len(anomalies)} ===')
for _, row in anomalies.iterrows():
    direction = '📈 HIGH' if z_scores[row.name] > 0 else '📉 LOW'
    print(f'{row["date"].date()}: Revenue ${row["revenue"]:.0f} ({direction}, z={z_scores[row.name]:.1f})')

## 5. 可视化

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Daily revenue by platform
daily = df.groupby(['date', 'platform'])['revenue'].sum().reset_index()
fig = px.line(daily, x='date', y='revenue', color='platform',
              title='Daily Revenue by Platform')
fig.show()

# Profit waterfall (Amazon)
amz_total = df[df['platform'] == 'amazon'].sum()
waterfall = go.Figure(go.Waterfall(
    x=['Revenue', 'COGS', 'FBA Fees', 'Ad Spend', 'Refunds', 'Net Profit'],
    y=[amz_total['revenue'], -amz_total['cogs'], -amz_total['fba_fees'],
       -amz_total['ad_spend'], -amz_total['refunds'], amz_total['net_profit']],
    measure=['absolute', 'relative', 'relative', 'relative', 'relative', 'total'],
    connector={'line': {'color': 'rgb(63, 63, 63)'}}
))
waterfall.update_layout(title='Amazon Profit Waterfall (90 days)')
waterfall.show()

# ROAS trend
amz_daily = df[df['platform'] == 'amazon'].copy()
amz_daily['roas_7d'] = amz_daily['ad_revenue'].rolling(7).sum() / amz_daily['ad_spend'].rolling(7).sum()
fig = px.line(amz_daily, x='date', y='roas_7d', title='Amazon 7-Day Rolling ROAS')
fig.add_hline(y=3, line_dash='dash', line_color='red', annotation_text='Target ROAS=3x')
fig.show()

## 6. 导出报告

In [ ]:
# Export
df.to_csv('dashboard_data.csv', index=False)
print('Data exported to dashboard_data.csv')
print('\nNext step: Use this data with Streamlit dashboard')
print('Run: streamlit run dashboard.py')
print('See: paths/b-developers/b8-ecommerce-dashboard.md for full code')